# 19 · Pipeline patterns (like a real job) (Spark)

A batch pipeline is more than one query. It is a chain of small, testable steps:

**read → add audit columns → clean → check quality → transform → reconcile → write (idempotent) → log**

Words used here:
- **run_id** = a unique id for one run of the job, saved on every row it writes (so a bad run can be found and removed)
- **quality gate** = checks that **stop** the job when the data is wrong
- **idempotent** = a rerun gives the same result (no duplicates)
- **lookback** = also re-process the last few days, to catch late-arriving rows

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Audit columns on read  ·  🟢 easy

Read `data/orders.tsv` with a schema and add `run_id` (`'run_20251231_01'`), `load_ts` (current timestamp) and `source_file` (the file name, from the hidden `_metadata` column).

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q2. A small, testable clean step  ·  🟢 easy

Write a function `clean_orders(df)` that lower-cases and trims `status`, drops rows with NULL `order_id` and drops exact duplicates. Print the row count before and after.

*Expected result: 104 rows · columns: `order_id`, `customer_id`, `order_date`, `amount`, `status`*

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q3. Unit test a transformation  ·  🟡 medium

Test `clean_orders` with a tiny hand-made input: one duplicate, one `' Completed '` status, one NULL order_id. Compare to the expected output with `assertDataFrameEqual`.

In [ ]:
# your answer here (DataFrame API)

## Q4. Quality gate  ·  🟡 medium

Write `quality_gate(df)` that checks: no NULL order_id, order_id unique, no negative amount. It returns the check results and **raises an error** if any check fails. Run it on clean orders (pass) and on raw orders (fail: catch and print).

In [ ]:
# your answer here (DataFrame API)

## Q5. Reconcile and write an audit row  ·  🟡 medium

After a transform (daily revenue of completed clean orders), reconcile source vs target (count of orders and sum of amount). Append one audit row (run_id, counts, sums, status) to `tmp/audit` (Parquet) and show the audit table.

In [ ]:
# your answer here (DataFrame API)

## Q6. Idempotent daily job  ·  🔴 hard

Write `run_job(run_date)` that takes the clean orders of **that day**, adds `run_id`, and writes them to `tmp/daily_orders` partitioned by `order_date` with **dynamic** partition overwrite. Run it **twice** for `2025-10-14` and once for `2025-11-03`. Show the row count per date: no duplicates.

In [ ]:
# your answer here (DataFrame API)

## Q7. Latest version per key (snapshot + delta)  ·  🟡 medium

`snapshot` = customers with `updated_at = 2024-12-31`. A `delta` arrives with 3 rows: customer 1 changed twice (2025-01-05 and 2025-01-09) and new customer 41. Build the new snapshot: the **latest** version of every customer. Show customers 1, 2, 41.

*Expected result: 3 rows · columns: `customer_id`, `name`, `city`, `signup_date`, `updated_at`*

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q8. Lookback window for late data  ·  🟢 easy

For run date `2025-12-28` with a **3-day lookback**, select the slice of orders from 2025-12-26 to 2025-12-28 (inclusive), computed from the run date. Show the dates and counts in the slice.

*Expected result: 2 rows · columns: `order_date`, `count`*

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()